In [ ]:
from pathlib import Path
from shutil import copy2

import numpy as np
import pandas as pd

from atlas.alignment import (
    initialize_zarr_alignment_df,
    pairwise_zarr_alignment,
    calculate_cumulative_shifts,
    correct_z_alignment_from_points,
)
from atlas.io import (
    StitchedZarrStore,
    apply_alignment_zarr,
    zarr_array_to_czi,
    rm_tree,
)


In [ ]:
# User input
AXIAL_PIXEL_MICRONS = 70
series_folder = Path(r"/path/to/section-set")
stitched_zarr_path = series_folder / "stitched_sections.zarr"

if not stitched_zarr_path.is_dir():
    raise FileNotFoundError(f"Stitched Zarr store not found: {stitched_zarr_path}")

# Opening the store checks the axes and makes its section records available.
store = StitchedZarrStore.open(stitched_zarr_path)
if "images" not in store.group:
    raise ValueError("The stitched Zarr store does not contain an images array")
# Validate the stored pixel sizes and keep the first section's value in microns.
pixel_size_checks = store.validate_section_pixel_sizes()
pix_size_micron = pixel_size_checks["Acquisition Pixel size"]["first_um"]
print(f"Using acquisition pixel size: {pix_size_micron} µm")

pixel_size = {"Value": pix_size_micron, "Axial": AXIAL_PIXEL_MICRONS, "Unit": "µm"}

In [ ]:
section_order = store.ordered_sections(order_path=series_folder / "correct_order.txt")
print(f"Selected {len(section_order)} sections for alignment")
print("First S-to-Z mappings:", [(r["atlas_section_number"], r["z_index"]) for r in section_order[:8]])


In [ ]:
# Initialize consecutive section pairs in the selected alignment order.
down_scale = 10
z_align_df = initialize_zarr_alignment_df(store, section_order, down_scale)
z_align_df


In [ ]:
output_path = series_folder.joinpath("alignment_results")
output_path.mkdir(parents=True, exist_ok=True)

# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_zarr_alignment(z_align_df, store)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# Save the alignment DataFrame for later analysis.
z_align_df_path = output_path.joinpath("z_alignment_results.pkl")
z_align_df.to_pickle(z_align_df_path)

z_align_df 

#TODO: implement handling of outliers in the alignment results, this can be done by looking at the current shift and then decide whichones looks weird.

In [ ]:
# this is just to show that values can be read and we dont need to re run the full alignment 
z_align_df = pd.read_pickle(z_align_df_path)
z_align_df

In [ ]:
# Save a downsampled aligned preview for visual inspection.
use_down_sample = True
zarr_array, zarr_path = apply_alignment_zarr(z_align_df, 
                                             store, 
                                             output_path=series_folder / "aligned_preview.zarr", 
                                             percentile_low=2.0, percentile_high=99.95, 
                                             use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

# only use this if you prefer ZEN over napari for the visual inspection

#czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
# Apply one manual correction cycle to the latest saved alignment.
# Select corresponding landmarks on the CURRENT downsampled aligned preview.
# Coordinates are (output Z index, Y, X) in preview pixels, not physical units,
# ATLAS S numbers, or master-store Z indices. Each moving section follows its
# fixed section in the alignment sequence. Multiple pairs are combined by median.
# For another cycle, inspect the regenerated preview and replace the landmarks;
# rerunning the same points would add the same residual correction again.
do_correction = False
if do_correction:
    z_align_df = pd.read_pickle(z_align_df_path)
    backup_path = z_align_df_path.with_name("z_alignment_results_before_manual_correction.pkl")
    if not backup_path.exists():
        copy2(z_align_df_path, backup_path)
    print(f"Automatic alignment baseline: {backup_path}")

    list_of_fix_points = [np.array([36, 483., 515.])]
    list_of_mov_points = [np.array([37, 482., 486.])]
    corrected_df = correct_z_alignment_from_points(
        z_align_df,
        list_of_fix_points,
        list_of_mov_points,
    )

    use_down_sample = True
    zarr_array, zarr_path = apply_alignment_zarr(
        corrected_df, store,
        output_path=series_folder / "aligned_preview.zarr",
        percentile_low=2.0, percentile_high=99.95,
        use_down_sample=use_down_sample,
    )

    # Save after preview generation succeeds. The next cycle and full-resolution
    # export will read these corrected shifts and cumulative ROIs.
    corrected_df.to_pickle(z_align_df_path)
    z_align_df = corrected_df
    print(f"Saved corrected alignment: {z_align_df_path}")
    print("Reload the updated preview before selecting landmarks for another cycle.")

    out_pixel_size = pixel_size.copy()
    out_pixel_size["Value"] = pixel_size["Value"] * down_scale
    end_str = "_ds_aligned"
    czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)


In [ ]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [ ]:
# Use the latest saved alignment, including any manual corrections.
z_align_df = pd.read_pickle(z_align_df_path)

use_down_sample = False
zarr_array, zarr_path = apply_alignment_zarr(z_align_df, store, output_path=series_folder / "aligned_full_resolution.zarr", percentile_low=1.0, percentile_high=99.9, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

# only if we need CZI, but now we can inspect using napari
#czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
#if zarr_path.exists():
#  rm_tree(zarr_path)